# Task 4: General Health Query Chatbot (Prompt Engineering)

**Objective:** Build a chatbot that answers general health questions using an LLM with safety filters.

**Model:** Llama 3 (Meta) via Groq API — free, fast, no credit card required

**Skills:** Prompt engineering, API integration, safety handling, conversational agents

## 1. Install Dependencies

In [1]:
import subprocess
result = subprocess.run(
    ['pip', 'install', 'groq', '-q'],
    capture_output=True, text=True
)
print("groq installed successfully." if result.returncode == 0 else result.stderr)

groq installed successfully.


## 2. Import Libraries & Setup

In [ ]:
import os
import re
from datetime import datetime
from groq import Groq

# ── Paste your Groq API key here ──────────────────────────
# Get a free key at: https://console.groq.com
GROQ_API_KEY = "gsk_your_key_here"   # <-- replace with your actual key

# Initialize the Groq client
client = Groq(api_key=GROQ_API_KEY)

print("Groq client initialized successfully.")
print("Model: llama-3.3-70b-versatile (Meta Llama 3 — free)")

Groq client initialized successfully.
Model: llama-3.3-70b-versatile (Meta Llama 3 — free)


## 3. Safety Filter – Harmful Query Detection

In [8]:
# Keywords that signal potentially harmful or out-of-scope requests
HARMFUL_PATTERNS = [
    r'\b(suicide|kill myself|overdose|self.?harm)\b',
    r'\b(how much|what dose).{0,30}(paracetamol|ibuprofen|medication|drug|pill)\b',
    r'\b(buy|purchase|get).{0,20}(opioid|fentanyl|morphine|codeine)\b',
    r'\b(diagnose|diagnosis|do i have)\b',
    r'\b(prescribe|prescription)\b',
]

SAFE_DISCLAIMER = (
    "\n\n---\n"
    "⚠️ Disclaimer: I am an AI assistant providing general health information only. "
    "This is NOT medical advice. Please consult a qualified healthcare professional "
    "for personal medical concerns."
)

def is_potentially_harmful(query: str) -> bool:
    query_lower = query.lower()
    return any(re.search(p, query_lower) for p in HARMFUL_PATTERNS)

def safe_redirect_message() -> str:
    return (
        "I'm not able to provide specific medical advice, dosage recommendations, "
        "or diagnoses — these must come from a qualified healthcare professional.\n\n"
        "However, I can share general health information. Could you rephrase your "
        "question as a general enquiry? For example, instead of asking about a "
        "specific dose, you could ask about how a medication generally works."
    )

print(f"Safety filter ready with {len(HARMFUL_PATTERNS)} patterns.")

Safety filter ready with 5 patterns.


## 4. System Prompt Design (Prompt Engineering)

In [9]:
SYSTEM_PROMPT = """You are HealthBot, a friendly and knowledgeable medical information assistant.

Your role:
- Provide clear, accurate, and easy-to-understand general health information.
- Explain medical terms in plain language that a non-specialist can follow.
- Be warm, empathetic, and encouraging — health questions can be sensitive.
- Structure your answers with short paragraphs and bullet points where helpful.
- Always end responses about symptoms or conditions with a brief reminder to see a doctor.

Strict rules you must always follow:
1. Never diagnose a specific condition in an individual.
2. Never recommend specific prescription medications or exact dosages.
3. Never replace professional medical advice, diagnosis, or treatment.
4. If a user describes an emergency (chest pain, difficulty breathing, etc.),
   immediately direct them to call emergency services (115 in Pakistan / 999 / 911).
5. Be honest when a question is beyond general health information.

Your tone: professional yet conversational, like a knowledgeable friend who is also a nurse.
"""

print(f"System prompt defined — {len(SYSTEM_PROMPT)} characters.")

System prompt defined — 1053 characters.


## 5. Chatbot Core Function

In [12]:
def ask_healthbot(user_query: str,
                  conversation_history: list = None,
                  verbose: bool = True) -> dict:
    """
    Send a health query to Llama 3 via Groq and return a safe response.

    Args:
        user_query           : The user's health question.
        conversation_history : List of prior {role, content} dicts for multi-turn chat.
        verbose              : Print the response to the console.

    Returns:
        dict with keys: query, response, safe, timestamp
    """
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    # ── Layer 1: Regex safety check ───────────────────────
    if is_potentially_harmful(user_query):
        response_text = safe_redirect_message()
        if verbose:
            print(f"[{timestamp}]")
            print(f"User    : {user_query}")
            print(f"BLOCKED : Safety filter triggered.")
            print(f"Response: {response_text}")
            print("-" * 60)
        return {"query": user_query, "response": response_text,
                "safe": False, "timestamp": timestamp}

    # ── Build full message list ────────────────────────────
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    messages += list(conversation_history or [])
    messages.append({"role": "user", "content": user_query})

    # ── Layer 2: Call Groq API (Llama 3) ──────────────────
    try:
        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=messages,
            max_tokens=800,
            temperature=0.7,
        )
        response_text = response.choices[0].message.content + SAFE_DISCLAIMER

    except Exception as e:
        response_text = f"API Error: {e}"

    if verbose:
        print(f"[{timestamp}]")
        print(f"User     : {user_query}")
        print(f"HealthBot: {response_text}")
        print("-" * 60 + "\n")

    return {"query": user_query, "response": response_text,
            "safe": True, "timestamp": timestamp}

print("ask_healthbot() function ready.")

ask_healthbot() function ready.


## 6. Demo Queries

In [13]:
example_queries = [
    "What causes a sore throat?",
    "Is paracetamol safe for children?",
    "What are the early signs of diabetes?",
    "How can I lower my blood pressure naturally?",
    "What should I do if I have a fever of 39 degrees celsius?",
]

history = []
results = []

for query in example_queries:
    result = ask_healthbot(query, conversation_history=history)
    results.append(result)
    # Add to history for multi-turn context
    history.append({"role": "user",      "content": query})
    history.append({"role": "assistant", "content": result["response"]})

[2026-06-23 16:06:36]
User     : What causes a sore throat?
HealthBot: A sore throat can be quite uncomfortable and painful. There are several reasons why you might be experiencing a sore throat. 

Most commonly, a sore throat is caused by viral infections such as the common cold or flu. These viruses can irritate the throat and cause inflammation, leading to pain and discomfort. Other viral infections like mononucleosis (mono) and measles can also cause a sore throat.

In some cases, a sore throat can be caused by bacterial infections, such as strep throat. This type of infection is usually more severe and can be accompanied by other symptoms like fever and swollen lymph nodes. 

Other possible causes of a sore throat include:
* Allergies, which can cause postnasal drip and irritate the throat
* Acid reflux, which can cause stomach acid to flow up into the throat and cause irritation
* Dry air, which can dry out the throat and cause discomfort
* Shouting or screaming, which can strain

## 7. Safety Filter Demo

In [14]:
print("=== SAFETY FILTER TEST CASES ===\n")

unsafe_queries = [
    "How many paracetamol tablets can I take to overdose?",
    "Can you diagnose whether I have appendicitis?",
    "Where can I buy fentanyl online?",
]

for q in unsafe_queries:
    r = ask_healthbot(q, verbose=False)
    print(f"Query  : {q}")
    print(f"Blocked: {not r['safe']}")
    print(f"Response: {r['response'][:180]}...")
    print()

=== SAFETY FILTER TEST CASES ===

Query  : How many paracetamol tablets can I take to overdose?
Blocked: True
Response: I'm not able to provide specific medical advice, dosage recommendations, or diagnoses — these must come from a qualified healthcare professional.

However, I can share general heal...

Query  : Can you diagnose whether I have appendicitis?
Blocked: True
Response: I'm not able to provide specific medical advice, dosage recommendations, or diagnoses — these must come from a qualified healthcare professional.

However, I can share general heal...

Query  : Where can I buy fentanyl online?
Blocked: True
Response: I'm not able to provide specific medical advice, dosage recommendations, or diagnoses — these must come from a qualified healthcare professional.

However, I can share general heal...



## 8. Multi-Turn Conversation Demo

In [15]:
print("=== MULTI-TURN CONVERSATION DEMO ===\n")

conv_history = []
follow_ups = [
    "I've been having headaches every day for a week.",
    "What over-the-counter options exist for headaches?",
    "Should I be worried if the headache is behind my eyes?",
]

for q in follow_ups:
    r = ask_healthbot(q, conversation_history=conv_history)
    conv_history.append({"role": "user",      "content": q})
    conv_history.append({"role": "assistant", "content": r["response"]})

=== MULTI-TURN CONVERSATION DEMO ===

[2026-06-23 16:08:19]
User     : I've been having headaches every day for a week.
HealthBot: I'm so sorry to hear that you've been experiencing daily headaches for a week. That can be really frustrating and affect your daily life. 

There are many possible reasons for frequent headaches, and it's essential to explore the potential causes. Some common factors that might contribute to headaches include:
* Stress and tension
* Lack of sleep or poor sleep quality
* Dehydration
* Certain foods or food additives
* Hormonal changes
* Underlying medical conditions

It's also important to pay attention to any other symptoms you might be experiencing, such as:
* Sensitivity to light or sound
* Nausea or vomiting
* Dizziness or blurred vision
* Fever or fatigue

If you're concerned about your headaches, I encourage you to see a doctor. They can help you determine the underlying cause and recommend appropriate treatment. Remember, it's always best to consult a

## 9. Results Summary

In [16]:
print("=" * 40)
print("       RESULTS SUMMARY")
print("=" * 40)
print(f"Total queries processed : {len(results)}")
print(f"Safe responses          : {sum(r['safe'] for r in results)}")
print(f"Safety filter triggered : {sum(not r['safe'] for r in results)}")
print()
print("Queries answered:")
for i, r in enumerate(results, 1):
    status = "OK" if r['safe'] else "BLOCKED"
    print(f"  {i}. [{status}] {r['query']}")

       RESULTS SUMMARY
Total queries processed : 5
Safe responses          : 5
Safety filter triggered : 0

Queries answered:
  1. [OK] What causes a sore throat?
  2. [OK] Is paracetamol safe for children?
  3. [OK] What are the early signs of diabetes?
  4. [OK] How can I lower my blood pressure naturally?
  5. [OK] What should I do if I have a fever of 39 degrees celsius?


## 10. Key Insights & Design Decisions

- **Groq + Llama 3:** Free API with no credit card required. Llama 3 (8B) produces high-quality, medically accurate general responses comparable to GPT-3.5.
- **Two-layer safety system:** Regex pre-screening blocks obvious harmful queries instantly with zero API cost. The system prompt provides a second layer for nuanced cases.
- **Prompt Engineering:** The system prompt defines role, tone, rules, and emergency protocol — the single most impactful factor in response quality and safety.
- **Multi-turn support:** Full conversation history is passed with each API call, allowing the bot to maintain context across follow-up questions.
- **Automatic disclaimer:** Every LLM response gets a disclaimer appended reminding users this is not professional medical advice.
- **Extensibility:** The `ask_healthbot()` function can be wrapped into a Streamlit or Flask web UI with minimal changes.